In [1]:
from importlib.metadata import version 

print ("torch version:", version("torch"))

torch version: 2.14.0


In [2]:
import torch

inputs = torch.tensor(
    [[0.43,0.15,0.89], # Your (x^1)
     [0.55,0.87,0.66], # journey (x^2)
     [0.57,0.85,0.64], # starts (x^3)
     [0.22,0.58,0.33], # with (x^4)
     [0.77,0.25,0.10], # one (x^5)
     [0.05,0.80,0.55] # step (x^6)
    ]
)


In [3]:
query = inputs[1] #(x^2)
print(query)

tensor([0.5500, 0.8700, 0.6600])


In [6]:
attn_scores_2 = torch.empty(inputs.shape[0]) # 计算attention score
for i, x_i in enumerate (inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [7]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum()) # normalization

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [8]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0) 
    #dim = 0 是按地0维压缩
    #torch.exp(x) . sum (1+2+3+....+n) ===》对前面的tensor sum
  

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [13]:
attn_weigths_2_softmax = torch.softmax(attn_scores_2,dim=0)
print("Attention weights:", attn_weigths_2_softmax)
print("Sum:", attn_weigths_2_softmax.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [15]:
#接下来计算 context vector 
#就是用每一个input x attention weight 再相加
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)

for i,x_i in enumerate(inputs):
    curr = attn_weigths_2_softmax[i]*x_i # 使用了emuerate 生成1 2 3 ... n 就是i  x_i 是里面原来的内容
    print("The current is:",curr)
    context_vec_2 += curr
    print("Context_vec_2 is:",context_vec_2)
    

print("Context_vec_2 is:",context_vec_2)

The current is: tensor([0.0596, 0.0208, 0.1233])
Context_vec_2 is: tensor([0.0596, 0.0208, 0.1233])
The current is: tensor([0.1308, 0.2070, 0.1570])
Context_vec_2 is: tensor([0.1904, 0.2277, 0.2803])
The current is: tensor([0.1330, 0.1983, 0.1493])
Context_vec_2 is: tensor([0.3234, 0.4260, 0.4296])
The current is: tensor([0.0273, 0.0719, 0.0409])
Context_vec_2 is: tensor([0.3507, 0.4979, 0.4705])
The current is: tensor([0.0833, 0.0270, 0.0108])
Context_vec_2 is: tensor([0.4340, 0.5250, 0.4813])
The current is: tensor([0.0079, 0.1265, 0.0870])
Context_vec_2 is: tensor([0.4419, 0.6515, 0.5683])
Context_vec_2 is: tensor([0.4419, 0.6515, 0.5683])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

# 这样我们就计算了context_vec_2

In [18]:
# 继续 我们要扩展context_vec_2 到所有的inputs

# 首先计算 attention scores

attn_scores = torch.empty(6,6) # 注意这里一定是 n，n 每一个都从其他的n个（包括自己）算一遍 
for i,x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i,x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [20]:
attn_weights = torch.softmax(attn_scores,dim = -1) #-1 最后一维度
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [21]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
